This notebook can be run on the Copernicus Dataspace Jupyterhub but running the following package installation cell first

**Note** You should select on of the kernels with GDAL installed, eg. "Geo science"

# Two-Source Energy Balance (TSEB) Model Implementation

This notebook implements the **Two-Source Energy Balance (TSEB) model** using the `pyTSEB` package.
It processes sharpened Sentinel-3 Land Surface Temperature (LST) data along with meteorological and vegetation parameters to compute land surface energy fluxes (including evapotranspiration) at high spatial resolution.

This notebook can be run on the [Copernicus Dataspace Jupyterhub](https://jupyterhub.dataspace.copernicus.eu) in which case data downloads are downloaded since both the data and the execution environment are on CDSE. 

**Note** You should select on of the kernels with GDAL installed, eg. "Geo science".

First we check that Sen-ET Toolbox is installed (and install it if necessary) and then we import all the necessary packages.

In [ ]:
try:
    import senet_toolbox
except ModuleNotFoundError:
    !pip install senet_toolbox@git+https://github.com/DHI/Sen-ET-OpenEO-toolbox.git

In [ ]:
import datetime as dt
from pathlib import Path
import re

from pyTSEB.PyTSEB import PyTSEB
from senet_toolbox import read_area_date_info, date_selector

## 1. Select Area of Interest
To keep the data organised, and make processing of timeseries easier, the input and output data is kept in Area of Interest (AOI) folders. All data within an AOI folder has the same extent and grid. 

In the cell below, select the data location and AOI name which was set in input data collection notebook. It will also pick the last date which was processed by that notebook. If you would like to work with a different date then use the second cell to list the dates within the AOI for which input data was collected, select the one you want to work with from the drop down list and run the third cell to save your choice. 

In [ ]:
data_dir = "./mystorage/"
aoi_name = "botswana"
aoi_data_dir = Path(data_dir) / aoi_name
date, bbox = read_area_date_info(dir=aoi_data_dir)
date = str(date)
date = date.replace("-", "")
date_data_dir = aoi_data_dir / date
output_file = date_data_dir / "output" / f"TSEB-PT_{date}.vrt"
print(date)

In [ ]:
date_selection = date_selector.get_collected_dates(
    aoi_data_dir = Path(data_dir) / aoi_name
)

In [ ]:
date = date_selection.value
date = date.replace("-", "")
date_data_dir = aoi_data_dir / date

lst_files = list(date_data_dir.glob("s3_*_LST_sharpened.tif"))
if not lst_files:
    raise FileNotFoundError(f"No sharpened Sentinel-3 LST found in {date_data_dir} run step 2 first")

match = re.search(r"s3_(\d{8}T\d{6})_", lst_files[0].name)
date_time = match.group(1)

print("Detected observation timestamp:", date_time)

# Convert timestamp
dt_obj = dt.datetime.strptime(date_time, "%Y%m%dT%H%M%S")

doy = dt_obj.timetuple().tm_yday
time_utc = dt_obj.hour + dt_obj.minute/60 + dt_obj.second/3600

print("Day of Year:", doy)
print("UTC time:", time_utc)

# Output path
output_file = date_data_dir / "output" / f"TSEB-PT_{date}.vrt"

### Define TSEB model parameters
Here you can specify which input data, and other parameterization, the TSEB model should use. You can leave it as it is if you where running the other notebooks with default values and would like to use the default TSEB parameterization.

In [ ]:
params = {
    "model": "TSEB_PT",
    "output_file": str(output_file),
    "T_R1": str(date_data_dir / f"s3_{date_time}_LST_sharpened.tif"),  # land surface temperature - this should be the Sharpened Sentinel-3 LST
    "VZA": str(date_data_dir / f"s3_{date_time}_VZA.tif"),
    "input_mask": 1,
    "LAI": str(date_data_dir / f"{date}_LAI.tif"),
    "f_c": str(aoi_data_dir / "F_C.tif"),
    "h_C": str(date_data_dir / f"{date}_H_C.tif"),
    "w_C": str(aoi_data_dir / "W_C.tif"),
    "f_g": str(date_data_dir / f"{date}_F_G.tif"),
    "lat": str(aoi_data_dir / "lat.tif"),
    "lon": str(aoi_data_dir / "lon.tif"),
    "alt": str(aoi_data_dir / f"cdem.tif"),
    "stdlon": 0,
    "z_T": 100,
    "z_u": 100,
    "DOY": doy,
    "time": time_utc,
    "T_A1": str(date_data_dir / f"{date_time}_TA.tif"),
    "u": str(date_data_dir / f"{date_time}_WS.tif"),
    "p": str(date_data_dir / f"{date_time}_PA.tif"),
    "ea": str(date_data_dir / f"{date_time}_EA.tif"),
    "S_dn": str(date_data_dir / f"{date_time}_SW-IN.tif"),
    "S_dn_24": str(date_data_dir / f"{date}_SW-IN-DD.tif"),
    "emis_C": 0.99,
    "emis_S": 0.97,
    "tau_vis_C": str(date_data_dir / f"{date}_TAU_VIS_C.tif"),
    "rho_vis_C": str(date_data_dir / f"{date}_RHO_VIS_C.tif"),
    "rho_nir_C": str(date_data_dir / f"{date}_RHO_NIR_C.tif"),
    "tau_nir_C": str(date_data_dir / f"{date}_TAU_NIR_C.tif"),
    "rho_vis_S": 0.15,
    "rho_nir_S": 0.25,
    "alpha_PT": 1.26,
    "x_LAD": str(aoi_data_dir / "X_LAD.tif"),
    "z0_soil": 0.01,
    "landcover": str(aoi_data_dir / "IGBP.tif"),
    "leaf_width": str(aoi_data_dir/ "LEAF_WIDTH.tif"),
    "resistance_form": 0,
    "KN_b": 0.012,
    "KN_c": 0.0038,
    "KN_C_dash": 90,
    "G_form": [[1], 0.35], # Constant ratio of 35% - ok when LST acquisition time is around noon
    "water_stress": 0,
    "calc_row": [1, 90],
    "row_az": 90,
}

### Run the TSEB model

Finally we can run the TSEB evapotranspiration model. The `ET_day` output file represents the daily evapotranspiration in units of mm/day. 

In [ ]:
model = PyTSEB(params)
results = model.process_local_image()

In [ ]:
visualization.show_raster_map(
    output_file,
    band="ET_day",
    cmap="YlGnBu"
)